In [ ]:
#|default_exp _graph

In [ ]:
#|hide
from nblite import nbl_export; nbl_export();

In [ ]:
#|export
import json
import sys
from pathlib import Path
from typing import Annotated, Any, Optional

import typer

from netrun_cli._helpers import (
    ConfigOpt,
    PrettyOpt,
    find_config,
    load_raw_data,
    output_json,
    write_config_data,
    deep_merge,
    auto_position,
    validate_after_write,
)

# Graph Management CLI Commands

Commands for adding, removing, and editing nodes and edges in netrun config files.
All mutating commands operate on raw dicts (not Pydantic models) to preserve
arbitrary extra fields and work on partially invalid configs.

## add-node

In [ ]:
#|export
NoValidateOpt = Annotated[bool, typer.Option("--no-validate", help="Skip post-write validation.")]


def add_node(
    name: Annotated[str, typer.Argument(help="Name for the new node.")],
    config: ConfigOpt = None,
    factory: Annotated[Optional[str], typer.Option("--factory", "-f", help="Factory import path.")] = None,
    factory_arg: Annotated[Optional[list[str]], typer.Option("--factory-arg", help="Factory arg as key=value (repeatable).")] = None,
    in_ports: Annotated[Optional[str], typer.Option("--in-ports", help="Comma-separated input port names.")] = None,
    out_ports: Annotated[Optional[str], typer.Option("--out-ports", help="Comma-separated output port names.")] = None,
    position: Annotated[Optional[str], typer.Option("--position", help="UI position as x,y.")] = None,
    from_json: Annotated[bool, typer.Option("--json", help="Read full node dict from stdin.")] = False,
    pretty: PrettyOpt = True,
    no_validate: NoValidateOpt = False,
) -> None:
    """Add a node to the graph."""
    data, config_path = load_raw_data(config)
    graph = data.setdefault("graph", {})
    nodes_list: list[dict] = graph.setdefault("nodes", [])

    # Check name uniqueness
    for existing in nodes_list:
        if existing.get("name") == name:
            typer.echo(f"Error: node '{name}' already exists.", err=True)
            raise typer.Exit(1)

    # Build node dict
    if from_json:
        raw_input = sys.stdin.read()
        try:
            node_dict = json.loads(raw_input)
        except json.JSONDecodeError as e:
            typer.echo(f"Error: invalid JSON from stdin: {e}", err=True)
            raise typer.Exit(1)
        node_dict["name"] = name
    else:
        node_dict: dict[str, Any] = {"name": name}
        if factory:
            node_dict["factory"] = factory
        if factory_arg:
            fa: dict[str, Any] = {}
            for item in factory_arg:
                if "=" not in item:
                    typer.echo(f"Error: --factory-arg must be key=value, got: {item}", err=True)
                    raise typer.Exit(1)
                k, v = item.split("=", 1)
                fa[k] = v
            node_dict["factory_args"] = fa
        if in_ports:
            node_dict["in_ports"] = {p.strip(): {} for p in in_ports.split(",")}
        if out_ports:
            node_dict["out_ports"] = {p.strip(): {} for p in out_ports.split(",")}

    # Position
    if position:
        parts = position.split(",")
        if len(parts) != 2:
            typer.echo("Error: --position must be x,y", err=True)
            raise typer.Exit(1)
        pos = {"x": float(parts[0]), "y": float(parts[1])}
    else:
        pos = auto_position(nodes_list)
    node_dict.setdefault("extra", {}).setdefault("ui", {})["position"] = pos

    nodes_list.append(node_dict)
    write_config_data(data, config_path)

    if not no_validate:
        validate_after_write(config_path)

    output_json(node_dict, pretty)

## remove-node

In [ ]:
#|export
def _edge_references_node(edge: dict, node_name: str) -> bool:
    """Check if an edge references a node."""
    if edge.get("source_node") == node_name:
        return True
    if edge.get("target_node") == node_name:
        return True
    return False


def remove_node(
    name: Annotated[str, typer.Argument(help="Name of the node to remove.")],
    config: ConfigOpt = None,
    pretty: PrettyOpt = True,
    no_validate: NoValidateOpt = False,
) -> None:
    """Remove a node and its connected edges from the graph."""
    data, config_path = load_raw_data(config)
    graph = data.get("graph", {})
    nodes_list: list[dict] = graph.get("nodes", [])

    # Find and remove node
    found = False
    for i, n in enumerate(nodes_list):
        if n.get("name") == name:
            nodes_list.pop(i)
            found = True
            break

    if not found:
        typer.echo(f"Error: node '{name}' not found.", err=True)
        raise typer.Exit(1)

    # Remove connected edges
    edges: list[dict] = graph.get("edges", [])
    original_edge_count = len(edges)
    graph["edges"] = [e for e in edges if not _edge_references_node(e, name)]
    edges_removed = original_edge_count - len(graph["edges"])

    # Clean output_queues that reference this node
    oq = data.get("output_queues")
    if isinstance(oq, dict):
        keys_to_remove = []
        for qname, qcfg in oq.items():
            if isinstance(qcfg, dict):
                ports = qcfg.get("ports", [])
                qcfg["ports"] = [p for p in ports if not _port_ref_matches_node(p, name)]
                if not qcfg["ports"]:
                    keys_to_remove.append(qname)
            elif isinstance(qcfg, str) and qcfg.startswith(f"{name}."):
                keys_to_remove.append(qname)
        for k in keys_to_remove:
            del oq[k]

    write_config_data(data, config_path)

    if not no_validate:
        validate_after_write(config_path)

    output_json({"removed": name, "edges_removed": edges_removed}, pretty)


def _port_ref_matches_node(port_ref: Any, node_name: str) -> bool:
    """Check if a port reference (string or dict) refers to a given node."""
    if isinstance(port_ref, str):
        return port_ref.split(".")[0] == node_name
    if isinstance(port_ref, dict):
        return port_ref.get("node_name") == node_name
    return False

## edit-node

In [ ]:
#|export
def _update_edge_node_refs(edges: list[dict], old_name: str, new_name: str) -> None:
    """Rename node references in edges."""
    for edge in edges:
        if edge.get("source_node") == old_name:
            edge["source_node"] = new_name
        if edge.get("target_node") == old_name:
            edge["target_node"] = new_name


def edit_node(
    name: Annotated[str, typer.Argument(help="Name of the node to edit.")],
    config: ConfigOpt = None,
    rename: Annotated[Optional[str], typer.Option("--rename", help="Rename the node.")] = None,
    add_in_port: Annotated[Optional[list[str]], typer.Option("--add-in-port", help="Add input port (repeatable).")] = None,
    remove_in_port: Annotated[Optional[list[str]], typer.Option("--remove-in-port", help="Remove input port (repeatable).")] = None,
    add_out_port: Annotated[Optional[list[str]], typer.Option("--add-out-port", help="Add output port (repeatable).")] = None,
    remove_out_port: Annotated[Optional[list[str]], typer.Option("--remove-out-port", help="Remove output port (repeatable).")] = None,
    merge: Annotated[Optional[str], typer.Option("--merge", help="JSON string to deep-merge into the node.")] = None,
    merge_stdin: Annotated[bool, typer.Option("--merge-stdin", help="Read JSON from stdin and deep-merge into the node.")] = False,
    pretty: PrettyOpt = True,
    no_validate: NoValidateOpt = False,
) -> None:
    """Edit an existing node in the graph."""
    data, config_path = load_raw_data(config)
    graph = data.get("graph", {})
    nodes_list: list[dict] = graph.get("nodes", [])

    # Find node
    node_dict = None
    for n in nodes_list:
        if n.get("name") == name:
            node_dict = n
            break
    if node_dict is None:
        typer.echo(f"Error: node '{name}' not found.", err=True)
        raise typer.Exit(1)

    # Rename
    if rename:
        # Check uniqueness
        for n in nodes_list:
            if n.get("name") == rename:
                typer.echo(f"Error: node '{rename}' already exists.", err=True)
                raise typer.Exit(1)
        old_name = node_dict["name"]
        node_dict["name"] = rename
        # Update edge references
        _update_edge_node_refs(graph.get("edges", []), old_name, rename)
        # Update output_queue refs
        oq = data.get("output_queues")
        if isinstance(oq, dict):
            for qcfg in oq.values():
                if isinstance(qcfg, dict):
                    ports = qcfg.get("ports", [])
                    for i, p in enumerate(ports):
                        if isinstance(p, str) and p.split(".")[0] == old_name:
                            ports[i] = rename + p[len(old_name):]
                        elif isinstance(p, dict) and p.get("node_name") == old_name:
                            p["node_name"] = rename

    # Port modifications
    if add_in_port:
        ip = node_dict.setdefault("in_ports", {})
        for p in add_in_port:
            ip[p] = {}
    if remove_in_port:
        ip = node_dict.get("in_ports", {})
        for p in remove_in_port:
            ip.pop(p, None)
    if add_out_port:
        op = node_dict.setdefault("out_ports", {})
        for p in add_out_port:
            op[p] = {}
    if remove_out_port:
        op = node_dict.get("out_ports", {})
        for p in remove_out_port:
            op.pop(p, None)

    # Merge
    if merge:
        try:
            merge_data = json.loads(merge)
        except json.JSONDecodeError as e:
            typer.echo(f"Error: invalid --merge JSON: {e}", err=True)
            raise typer.Exit(1)
        merged = deep_merge(node_dict, merge_data)
        # Replace in-place
        node_dict.clear()
        node_dict.update(merged)
    if merge_stdin:
        raw_input = sys.stdin.read()
        try:
            merge_data = json.loads(raw_input)
        except json.JSONDecodeError as e:
            typer.echo(f"Error: invalid JSON from stdin: {e}", err=True)
            raise typer.Exit(1)
        merged = deep_merge(node_dict, merge_data)
        node_dict.clear()
        node_dict.update(merged)

    write_config_data(data, config_path)

    if not no_validate:
        validate_after_write(config_path)

    output_json(node_dict, pretty)

## add-edge

In [ ]:
#|export
def add_edge(
    source_node_arg: Annotated[str, typer.Argument(help="Source node name.")],
    source_port_arg: Annotated[str, typer.Argument(help="Source port name.")],
    target_node_arg: Annotated[str, typer.Argument(help="Target node name.")],
    target_port_arg: Annotated[str, typer.Argument(help="Target port name.")],
    config: ConfigOpt = None,
    dependency: Annotated[bool, typer.Option("--dependency", help="Mark edge as a dependency edge.")] = False,
    pretty: PrettyOpt = True,
    no_validate: NoValidateOpt = False,
) -> None:
    """Add an edge between two ports."""
    data, config_path = load_raw_data(config)
    graph = data.setdefault("graph", {})
    nodes_list: list[dict] = graph.get("nodes", [])
    edges: list[dict] = graph.setdefault("edges", [])

    node_names = {n.get("name") for n in nodes_list}

    if source_node_arg not in node_names:
        typer.echo(f"Error: source node '{source_node_arg}' not found.", err=True)
        raise typer.Exit(1)
    if target_node_arg not in node_names:
        typer.echo(f"Error: target node '{target_node_arg}' not found.", err=True)
        raise typer.Exit(1)

    # Warn on fan-out
    for e in edges:
        if e.get("source_node") == source_node_arg and e.get("source_port") == source_port_arg:
            typer.echo(
                f"Warning: fan-out — '{source_node_arg}.{source_port_arg}' already has an edge. "
                "Consider using 'netrun.node_factories.broadcast' to replicate output to multiple targets.",
                err=True,
            )
            break

    edge_dict: dict[str, Any] = {
        "source_node": source_node_arg,
        "source_port": source_port_arg,
        "target_node": target_node_arg,
        "target_port": target_port_arg,
    }
    if dependency:
        edge_dict["dependency"] = True

    edges.append(edge_dict)
    write_config_data(data, config_path)

    if not no_validate:
        validate_after_write(config_path)

    output_json(edge_dict, pretty)

## remove-edge

In [ ]:
#|export
def _edge_matches(edge: dict, source_node: str, source_port: str, target_node: str, target_port: str) -> bool:
    """Check if an edge matches the given source and target fields."""
    return (
        edge.get("source_node") == source_node
        and edge.get("source_port") == source_port
        and edge.get("target_node") == target_node
        and edge.get("target_port") == target_port
    )


def remove_edge(
    source_node_arg: Annotated[str, typer.Argument(help="Source node name.")],
    source_port_arg: Annotated[str, typer.Argument(help="Source port name.")],
    target_node_arg: Annotated[str, typer.Argument(help="Target node name.")],
    target_port_arg: Annotated[str, typer.Argument(help="Target port name.")],
    config: ConfigOpt = None,
    pretty: PrettyOpt = True,
    no_validate: NoValidateOpt = False,
) -> None:
    """Remove an edge between two ports."""
    data, config_path = load_raw_data(config)
    graph = data.get("graph", {})
    edges: list[dict] = graph.get("edges", [])

    found = False
    for i, e in enumerate(edges):
        if _edge_matches(e, source_node_arg, source_port_arg, target_node_arg, target_port_arg):
            edges.pop(i)
            found = True
            break

    if not found:
        typer.echo(
            f"Error: edge '{source_node_arg}.{source_port_arg}' -> "
            f"'{target_node_arg}.{target_port_arg}' not found.",
            err=True,
        )
        raise typer.Exit(1)

    write_config_data(data, config_path)

    if not no_validate:
        validate_after_write(config_path)

    output_json({
        "removed_source_node": source_node_arg,
        "removed_source_port": source_port_arg,
        "removed_target_node": target_node_arg,
        "removed_target_port": target_port_arg,
    }, pretty)